## Preprocessing danych Synthea

### 📋 Cel notebooka
Ten notebook przetwarza dane syntetyczne z Synthea, generując sekwencje tokenów reprezentujące pacjentów i ich etykiety triage.

### 🔄 Wymagania
- `data/data/batch_*/csv/` - pliki CSV z danymi Synthea
- `config.py` - konfiguracja ścieżek i mapowań

### 📊 Metodologia

#### 1. Format danych wejściowych
- Pliki CSV z Synthea: `conditions.csv`, `observations.csv`, `patients.csv`
- Dane zawierają: wiek, płeć, diagnozy, wyniki badań laboratoryjnych

#### 2. Generowanie sekwencji
- **Tokeny wiekowe:** `AGE_0`, `AGE_10`, `AGE_20`, ..., `AGE_90+`
- **Tokeny płci:** `GENDER_M`, `GENDER_F`
- **Tokeny badań:** Kwantyzacja wyników do przedziałów Q1-Q10 (np. `HGB_Q5`, `HGB_Q1_CRITICAL`)
- **Composite features:** Kombinacje tokenów reprezentujące kontekst kliniczny (np. `COMPOSITE_ANEMIA_INDICATOR` = HGB niski + MCV niski)
- **Tokeny objawów:** Mapowanie diagnoz na tokeny objawów (np. `SYMPTOM_GI_BLEED_YES`)
- **Etykiety:** Mapowanie diagnoz na kategorie triage (`TARGET_POZ`, `TARGET_SPEC_GASTROLOG`, ...)

#### 3. Kwantyzacja wyników laboratoryjnych
- **Metoda:** Porównanie wartości z normami wiekowymi/płciowymi
- **Przedziały:** Q1-Q10 (Q1=poniżej normy, Q5=norma, Q10=powyżej normy)
- **Krytyczne wartości:** Q1_CRITICAL, Q10_CRITICAL (poza zakresem bezpieczeństwa)

#### 4. Mapowanie diagnoz
- Diagnozy są mapowane na kategorie triage zgodnie z `DIAGNOSIS_MAP` i `PRIORITY_MAP`
- Priorytety: POZ (najniższy), specjalista (średni), SOR (najwyższy)

### 📝 Output
- `synthea_data_YYYYMMDD_HHMMSS.txt` - przetworzone sekwencje z etykietami

In [ ]:
!pip install polars

In [ ]:
# ⚠️ WAŻNE: Ustawienie zmiennej środowiskowej dla polars (Rosetta x86_64)
import os
os.environ['POLARS_SKIP_CPU_CHECK'] = '1'  # Pomija sprawdzanie CPU features (Rosetta)

# Sprawdź i zainstaluj polars jeśli potrzeba
import sys
import warnings
warnings.filterwarnings('ignore')  # Ignoruj ostrzeżenia CPU (Mac ARM)

try:
    import polars as pl
    print(f"✅ polars już zainstalowana: {pl.__version__}")
    print(f"✅ POLARS_SKIP_CPU_CHECK={os.environ.get('POLARS_SKIP_CPU_CHECK', 'NOT SET')}")
except ImportError:
    print("📦 Instalowanie polars...")
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "polars[rtcompat]"])
    import polars as pl
    print(f"✅ polars zainstalowana: {pl.__version__}")

print(f"🐍 Python: {sys.executable}")

In [ ]:
# ⚠️ WAŻNE: Ustawienie zmiennej środowiskowej dla polars (Rosetta x86_64)
import os
os.environ['POLARS_SKIP_CPU_CHECK'] = '1'  # Pomija sprawdzanie CPU features (Rosetta)

import sys
from pathlib import Path
import json
import glob
import warnings
warnings.filterwarnings('ignore')  # Ignoruj ostrzeżenia CPU

print("📦 Importowanie bibliotek...")

try:
    import polars as pl
    print(f"✅ polars: {pl.__version__}")
    print(f"✅ POLARS_SKIP_CPU_CHECK={os.environ.get('POLARS_SKIP_CPU_CHECK', 'NOT SET')}")
except ImportError:
    print("❌ BŁĄD: polars nie jest zainstalowana!")
    print("   Uruchom Cell 1 najpierw, lub: pip install polars[rtcompat]")
    raise

import random
print("✅ random: OK")

# Add project root to sys.path
PROJECT_ROOT = Path(".").resolve().parent
print(f"📂 PROJECT_ROOT: {PROJECT_ROOT}")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))
    print(f"✅ Dodano {PROJECT_ROOT} do sys.path")

print("📥 Importowanie config...")
try:
    import config
    print("✅ Import config.py: SUKCES")
except Exception as e:
    print(f"❌ BŁĄD importu config: {e}")
    print(f"   Sprawdź, czy config.py istnieje w: {PROJECT_ROOT}")
    raise

# Using config paths
DATA_PATH = config.DATA_DIR
BATCHES_DIR = DATA_PATH / "raw" / "synthea" / "data"
QUESTIONS_FILE_PATH = config.QUESTIONS_PATH
LAB_NORMS_FILE_PATH = config.LAB_NORMS_PATH
LAB_CODES = config.LAB_CODES
DIAGNOSIS_MAP = config.DIAGNOSIS_MAP
PRIORITY_MAP = config.PRIORITY_MAP

print("\n✅ Konfiguracja załadowana:")
print(f"   DATA_PATH: {DATA_PATH}")
print(f"   BATCHES_DIR: {BATCHES_DIR}")
print(f"   Czy BATCHES_DIR istnieje: {BATCHES_DIR.exists()}")
if BATCHES_DIR.exists():
    batches = list(BATCHES_DIR.glob("batch_*"))
    print(f"   Znaleziono {len(batches)} batchy")
print(f"   QUESTIONS_FILE: {QUESTIONS_FILE_PATH} ({'✅' if QUESTIONS_FILE_PATH.exists() else '❌'})")
print(f"   LAB_NORMS_FILE: {LAB_NORMS_FILE_PATH} ({'✅' if LAB_NORMS_FILE_PATH.exists() else '❌'})")
print(f"   LAB_CODES: {len(LAB_CODES)} kodów")
print(f"   DIAGNOSIS_MAP: {len(DIAGNOSIS_MAP)} mapowań")
print(f"   PRIORITY_MAP: {len(PRIORITY_MAP)} priorytetów")


In [ ]:
def load_lab_norms(filepath):
    try:
        with open(filepath, "r") as f:
            return json.load(f)
    except FileNotFoundError:
        print("⚠️ Brak pliku lab_norms.json! Używam pustych norm.")
        return {}
    
# --- WKLEJ TO DO 01 i 02 ---

def get_lab_token_v2(test_name, val, age, sex, norms_db):
    if val is None: return None
    
    # 1. Pobieramy normy
    rules = norms_db.get(test_name)
    if not rules: return f"{test_name}_VAL"
    
    active_rule = None
    if isinstance(rules, list):
        for rule in rules:
            if "min_age" in rule and age < rule["min_age"]: continue
            if "max_age" in rule and age > rule["max_age"]: continue
            if "sex" in rule and rule["sex"] != sex: continue
            active_rule = rule
            break
        if not active_rule:
            active_rule = next((r for r in rules if "min_age" not in r and "sex" not in r), rules[-1])
    else:
        active_rule = rules
    
    # 2. Wyciągamy granice
    low = active_rule.get("low", -float('inf'))
    high = active_rule.get("high", float('inf'))
    crit_low = active_rule.get("critical_low", low * 0.6) # Domyślnie 60% normy
    crit_high = active_rule.get("critical_high", high * 1.5) # Domyślnie 150% normy
    
    # 3. KWANTYZACJA (Heurystyczna Q1-Q10 na podstawie norm)
    # To symuluje rozkład populacyjny względem norm medycznych
    
    token_suffix = ""
    is_critical = False
    
    if val < low:
        # Poniżej normy (Q1-Q3)
        if val < crit_low:
            token_suffix = "Q1_CRITICAL" # Ekstremalnie nisko
            is_critical = True
        elif val < low * 0.85:
            token_suffix = "Q2" # Bardzo nisko
        else:
            token_suffix = "Q3" # Trochę nisko
    elif val > high:
        # Powyżej normy (Q8-Q10)
        if val > crit_high:
            token_suffix = "Q10_CRITICAL" # Ekstremalnie wysoko
            is_critical = True
        elif val > high * 1.15:
            token_suffix = "Q9" # Bardzo wysoko
        else:
            token_suffix = "Q8" # Trochę wysoko
    else:
        # W normie (Q4-Q7)
        # Dzielimy zakres normy na 4 równe części
        step = (high - low) / 4
        if val < low + step: token_suffix = "Q4"
        elif val < low + 2*step: token_suffix = "Q5"
        elif val < low + 3*step: token_suffix = "Q6"
        else: token_suffix = "Q7"
        
    return f"{test_name}_{token_suffix}"

# --- POMOCNICZA FUNKCJA DO PYTAŃ (DLA SYNTHEA 01) ---
# Musi umieć przetłumaczyć Q1/Q2 na "LOW" żeby system pytań działał

def map_token_to_trigger(token):
    # Zamienia np. HGB_Q1_CRITICAL -> HGB_LOW
    if "Q1" in token or "Q2" in token or "Q3" in token:
        return token.split("_")[0] + "_LOW"
    if "Q8" in token or "Q9" in token or "Q10" in token:
        return token.split("_")[0] + "_HIGH"
    return None

def get_age_group(age):
    if age < 18: return "kids"
    if age < 30: return "under_30"
    if age < 60: return "under_60"
    return "seniors"

print("🚀 Start przetwarzania...")

# 1. Wczytanie Pytań
if not QUESTIONS_FILE_PATH.exists():
    # Fallback: Jeśli pliku nie ma, użyj pustego słownika lub stwórz go
    print(f"⚠️ Nie znaleziono {QUESTIONS_FILE_PATH}, używam pustej bazy pytań.")
    QUESTIONS_DB = {}
else:
    with open(QUESTIONS_FILE_PATH, "r") as f:
        QUESTIONS_DB = json.load(f)
    print("✅ Baza pytań wczytana.")

LAB_NORMS_DB = load_lab_norms(LAB_NORMS_FILE_PATH)
print("✅ Baza norm laboratoryjnych wczytana.")

# 2. Budowanie Wyroczni (Oracle) - Historia chorób
# Szukamy plików w: .../data/synthea/data/batch_*/csv/conditions.csv
search_pattern_cond = str(BATCHES_DIR / "batch_*" / "csv" / "conditions.csv")
cond_files = glob.glob(search_pattern_cond)

if not cond_files:
    raise FileNotFoundError(f"❌ Nie znaleziono plików conditions.csv w: {search_pattern_cond}")

print(f"📖 Budowanie mapy chorób z {len(cond_files)} plików...")

# Pobieramy CODE (dla pytań) oraz DESCRIPTION (dla Targetu/Diagnozy)
q_cond = (
    pl.scan_csv(cond_files)
    .select(["PATIENT", "CODE", "DESCRIPTION"])
    .group_by("PATIENT")
    .agg([
        pl.col("CODE").alias("ALL_CODES"),
        pl.col("DESCRIPTION").str.to_lowercase().alias("ALL_DESCS") # Opisy małymi literami
    ])
)

df_cond = q_cond.collect()

In [ ]:
# ORACLE_MAP trzyma dwie rzeczy: zbiór kodów (do pytań) i listę opisów (do diagnozy)
ORACLE_MAP = {}
for row in df_cond.to_dicts():
    ORACLE_MAP[row["PATIENT"]] = {
        "codes": set(str(c) for c in row["ALL_CODES"]),
        "descs": row["ALL_DESCS"]
    }

print(f"✅ Wyrocznia gotowa dla {len(ORACLE_MAP)} pacjentów.")

# 3. Agregacja Wyników Badań
search_pattern_obs = str(BATCHES_DIR / "batch_*" / "csv" / "observations.csv")
obs_files = glob.glob(search_pattern_obs)

print("🧪 Agregacja wyników badań...")
q_obs = (
    pl.scan_csv(obs_files)
    .filter(pl.col("CODE").is_in(LAB_CODES.keys()))
    .with_columns(
        pl.col("VALUE").cast(pl.Float64, strict=False).alias("VAL_NUM"),
        pl.col("CODE").replace(LAB_CODES).alias("TEST_NAME")
    )
    .drop_nulls("VAL_NUM")
    .group_by(["PATIENT", "TEST_NAME"])
    .agg(pl.col("VAL_NUM").mean())
    .collect()
    .pivot(index="PATIENT", on="TEST_NAME", values="VAL_NUM", aggregate_function="first")
)

In [ ]:
# --- SKRYPT POMOCNICZY: Znajdź Kody Chorób (Proxies) ---
# Wklej to do nowej komórki w 01_Preprocessing_Synthea.ipynb
import polars as pl

# Słowa kluczowe dla brakujących objawów
keywords = {
    "VOMIT/GI": ["gastritis", "ulcer", "reflux", "colitis", "diverticulosis", "appendicitis"],
    "KIDNEY/URINE": ["kidney", "renal", "nephropathy", "cystitis", "urinary"],
    "LIVER/JAUNDICE": ["liver", "cirrhosis", "hepatitis", "gallstones", "cholecystitis"],
    "HEART/CHEST": ["heart", "coronary", "angina", "myocardial", "fibrillation"],
    "WEIGHT": ["obesity", "body mass", "nutrition"] # Czasem utrata wagi wynika z braku tych kodów, ale szukamy chorób
}

print("🔎 Szukanie kodów chorób pasujących do objawów...")

# Pobieramy unikalne pary Code-Description z df_cond (zakładając, że df_cond jest w pamięci)
# Jeśli df_cond nie jest dostępne, musisz je odtworzyć z cella 2
unique_conditions = (
    df_cond
    .explode(["ALL_CODES", "ALL_DESCS"])
    .unique(subset=["ALL_CODES"])
    .select(["ALL_CODES", "ALL_DESCS"])
    .collect() if isinstance(df_cond, pl.LazyFrame) else df_cond.explode(["ALL_CODES", "ALL_DESCS"]).unique(subset=["ALL_CODES"]).select(["ALL_CODES", "ALL_DESCS"])
)

found_map = {}

for category, keys in keywords.items():
    print(f"\n--- {category} ---")
    
    # Filtrowanie
    mask = unique_conditions["ALL_DESCS"].str.contains("(?i)" + "|".join(keys))
    filtered = unique_conditions.filter(mask)
    
    for row in filtered.iter_rows():
        code, desc = row[0], row[1]
        print(f"[{code}] {desc}")
        found_map[code] = desc

print("\n✅ Gotowe. Wybierz odpowiednie kody i podstaw do questions.json")

In [ ]:
# 4. Demografia i Merge
search_pattern_pat = str(BATCHES_DIR / "batch_*" / "csv" / "patients.csv")
pat_files = glob.glob(search_pattern_pat)

q_pat = (
    pl.scan_csv(pat_files)
    .select(["Id", "BIRTHDATE", "GENDER"])
    .rename({"Id": "PATIENT"})
    .with_columns([(pl.lit(2025) - pl.col("BIRTHDATE").str.slice(0, 4).cast(pl.Int32)).alias("AGE")])
)

df_final = q_pat.collect().join(q_obs, on="PATIENT", how="inner")
print(f"🔗 Połączono dane dla {len(df_final)} pacjentów.")

In [ ]:
PRIORITY_MAP = {
        'TARGET_SPEC_SOR': 100,      # Ratunkowe (z MIMIC)
        'TARGET_SPEC_NEFROLOG': 50,  # Nerki
        'TARGET_SPEC_GASTROLOG': 40, # Krwawienia
        'TARGET_SPEC_HEMATOLOG': 30, # Anemie
        'TARGET_POZ': 0
    }

In [ ]:
# 5. Generowanie Sekwencji
print("🧠 Aplikowanie logiki pytań i diagnoz (z obsługą Q1-Q10)...")
final_sequences = []
stats = {"TARGET_POZ": 0, "TARGET_SPEC_HEMATOLOG": 0, "TARGET_SPEC_NEFROLOG": 0, "TARGET_SPEC_GASTROLOG": 0}

# Helper do mapowania Q -> Trigger (żeby pytania działały)
def extract_triggers(tokens):
    triggers = set()
    for t in tokens:
        # Zakładamy format: TEST_SUFFIX np. HGB_Q1_CRITICAL
        parts = t.split('_')
        test_name = parts[0]
        
        # Logika mapowania kwantyli na LOW/HIGH
        if any(x in t for x in ["Q1", "Q2", "Q3", "LOW", "CRITICAL_LOW"]):
            triggers.add(f"{test_name}_LOW")
        if any(x in t for x in ["Q8", "Q9", "Q10", "HIGH", "CRITICAL_HIGH"]):
            triggers.add(f"{test_name}_HIGH")
            
        # Dodajemy też sam token krytyczny jako trigger (opcjonalnie)
        if "CRITICAL" in t:
            triggers.add(f"{test_name}_CRITICAL")
            
    return triggers

for row in df_final.to_dicts():
    pat_id = row["PATIENT"]
    age = row["AGE"]
    sex = row["GENDER"]
    
    # Dane o pacjencie z Wyroczni
    oracle_data = ORACLE_MAP.get(pat_id, {"codes": set(), "descs": []})
    patient_codes = oracle_data["codes"]
    patient_descs = oracle_data["descs"] 

    # Tokeny podstawowe
    tokens = [f"AGE_{int(age/10)*10}", f"SEX_{sex}"]
    
    # Lab tokeny (Teraz generuje np. HGB_Q1_CRITICAL zamiast HGB_LOW)
    lab_tokens = []
    
    # Parametry szumu (możesz sterować siłą psucia)
    # FIX: Wyłączamy baked-in noise. Szum robimy dynamicznie podczas treningu.
    DROP_PROB = 0.0  # Było 0.15
    NOISE_FACTOR = 0.0 # Było 0.05

    for test_name in LAB_CODES.values():
        if test_name in row and row[test_name] is not None:
            
            # 1. LAB DROPOUT: Losowo pomijamy badanie (tylko dla treningu!)
            # Dzięki temu model nauczy się wnioskować nawet przy braku np. UREA
            if random.random() < DROP_PROB:
                continue 
            
            # 2. VALUE JITTER: Dodajemy szum do wartości przed kwantyzacją
            original_val = row[test_name]
            # Mnożymy przez losową wartość z zakresu 0.95 - 1.05
            noisy_val = original_val * random.uniform(1.0 - NOISE_FACTOR, 1.0 + NOISE_FACTOR)
            
            # Generujemy token na podstawie zaszumionej wartości
            token = get_lab_token_v2(test_name, noisy_val, age, sex, LAB_NORMS_DB)
            
            if token:
                lab_tokens.append(token)

    tokens.extend(lab_tokens)
    
    # --- COMPOSITE FEATURES: Kontekst kliniczny ---
    # Dodajemy composite features na podstawie kombinacji tokenów
    # To pozwala modelowi lepiej rozumieć kontekst kliniczny (np. anemia = HGB_Q1 + MCV_Q1)
    
    def generate_composite_features(lab_tokens):
        """
        Generuje composite features na podstawie kombinacji tokenów lab.
        
        Composite features uwzględniają kontekst kliniczny:
        - ANEMIA_INDICATOR: HGB niski + MCV niski/normalny
        - THROMBOCYTOPENIA_INDICATOR: PLT niski
        - RENAL_FAILURE_INDICATOR: CREATININE wysoki + UREA wysoki
        - ACUTE_KIDNEY_INJURY: CREATININE wysoki + EGFR niski
        - BLEEDING_RISK: PLT niski + HGB niski
        - INFECTION_INDICATOR: WBC wysoki + inne markery
        """
        composite_tokens = []
        lab_tokens_set = set(lab_tokens)
        
        # 1. ANEMIA_INDICATOR = (HGB_Q1 OR HGB_Q2) AND (MCV_Q1 OR MCV_Q2 OR MCV_Q3)
        # Anemia: niska hemoglobina + niski/normalny MCV (mikrocytowa/normocytowa)
        hgb_low = any('HGB_Q1' in t or 'HGB_Q2' in t or 'HGB_Q3' in t for t in lab_tokens)
        mcv_low_normal = any('MCV_Q1' in t or 'MCV_Q2' in t or 'MCV_Q3' in t or 'MCV_Q4' in t or 'MCV_Q5' in t for t in lab_tokens)
        if hgb_low and mcv_low_normal:
            composite_tokens.append('COMPOSITE_ANEMIA_INDICATOR')
        
        # 2. THROMBOCYTOPENIA_INDICATOR = PLT_Q1 OR PLT_Q2
        # Małopłytkowość: niskie płytki krwi
        plt_low = any('PLT_Q1' in t or 'PLT_Q2' in t or 'PLT_Q3' in t for t in lab_tokens)
        if plt_low:
            composite_tokens.append('COMPOSITE_THROMBOCYTOPENIA_INDICATOR')
        
        # 3. RENAL_FAILURE_INDICATOR = (CREATININE_Q8 OR CREATININE_Q9 OR CREATININE_Q10) AND (UREA_Q8 OR UREA_Q9 OR UREA_Q10)
        # Niewydolność nerek: wysoka kreatynina + wysoki mocznik
        creatinine_high = any('CREATININE_Q8' in t or 'CREATININE_Q9' in t or 'CREATININE_Q10' in t for t in lab_tokens)
        urea_high = any('UREA_Q8' in t or 'UREA_Q9' in t or 'UREA_Q10' in t for t in lab_tokens)
        if creatinine_high and urea_high:
            composite_tokens.append('COMPOSITE_RENAL_FAILURE_INDICATOR')
        
        # 4. ACUTE_KIDNEY_INJURY = CREATININE wysoki + EGFR niski (jeśli dostępny)
        # Ostre uszkodzenie nerek
        egfr_low = any('EGFR_Q1' in t or 'EGFR_Q2' in t or 'EGFR_Q3' in t for t in lab_tokens)
        if creatinine_high and egfr_low:
            composite_tokens.append('COMPOSITE_ACUTE_KIDNEY_INJURY')
        
        # 5. BLEEDING_RISK = PLT niski + HGB niski
        # Ryzyko krwawienia: niskie płytki + niska hemoglobina
        if plt_low and hgb_low:
            composite_tokens.append('COMPOSITE_BLEEDING_RISK')
        
        # 6. INFECTION_INDICATOR = WBC wysoki
        # Infekcja: podwyższone białe krwinki
        wbc_high = any('WBC_Q8' in t or 'WBC_Q9' in t or 'WBC_Q10' in t for t in lab_tokens)
        if wbc_high:
            composite_tokens.append('COMPOSITE_INFECTION_INDICATOR')
        
        # 7. SEVERE_ANEMIA = HGB_Q1_CRITICAL (ekstremalnie niska hemoglobina)
        # Ciężka anemia: krytycznie niska hemoglobina
        hgb_critical = any('HGB_Q1_CRITICAL' in t for t in lab_tokens)
        if hgb_critical:
            composite_tokens.append('COMPOSITE_SEVERE_ANEMIA')
        
        # 8. RENAL_CRITICAL = CREATININE_Q10_CRITICAL (ekstremalnie wysoka kreatynina)
        # Krytyczna niewydolność nerek
        creatinine_critical = any('CREATININE_Q10_CRITICAL' in t for t in lab_tokens)
        if creatinine_critical:
            composite_tokens.append('COMPOSITE_RENAL_CRITICAL')
        
        return composite_tokens
    
    # Generuj composite features
    composite_tokens = generate_composite_features(lab_tokens)
    tokens.extend(composite_tokens)
    
    # --- WAŻNE: Wyciągamy "Triggery" z nowych tokenów ---
    # Żeby system wiedział, że HGB_Q1_CRITICAL oznacza HGB_LOW i zadał pytanie
    active_triggers = extract_triggers(lab_tokens)
    
    # --- A. PYTANIA ADAPTACYJNE ---
    age_group = get_age_group(age)
    rules = QUESTIONS_DB.get(age_group, [])
    
    for rule in rules:
        # TERAZ SPRAWDZAMY W ACTIVE_TRIGGERS, A NIE W LAB_TOKENS
        if rule["trigger"] in active_triggers:
            if "gender" in rule and rule["gender"] != sex: continue
            
            # Czy kod z pytania jest w historii pacjenta?
            has_condition = str(rule.get("snomed_code", "")) in patient_codes
            answer_token = rule["token_yes"] if has_condition else rule["token_no"]
            tokens.append(answer_token)
            
    # --- B. USTALANIE TARGETU ---
    target = "TARGET_POZ"
    
    candidates = set()
    for desc in patient_descs:
        for keyword, label in DIAGNOSIS_MAP.items():
            if keyword in desc:
                candidates.add(label)
    
    # Tutaj też używamy triggerów, żeby wyłapać CREATININE_HIGH (nawet jak jest Q10)
    if "CREATININE_HIGH" in active_triggers:
        candidates.add('TARGET_SPEC_NEFROLOG')
        
    if "SYMPTOM_GI_BLEED_YES" in tokens:
        candidates.add('TARGET_SPEC_GASTROLOG')
    tokens_set = set(tokens)
    if 'COMPOSITE_ANEMIA_INDICATOR' in tokens_set or 'COMPOSITE_SEVERE_ANEMIA' in tokens_set:
        candidates.add('TARGET_SPEC_HEMATOLOG')
    if 'COMPOSITE_RENAL_FAILURE_INDICATOR' in tokens_set or 'COMPOSITE_ACUTE_KIDNEY_INJURY' in tokens_set or 'COMPOSITE_RENAL_CRITICAL' in tokens_set:
        candidates.add('TARGET_SPEC_NEFROLOG')
    if 'COMPOSITE_BLEEDING_RISK' in tokens_set or 'COMPOSITE_THROMBOCYTOPENIA_INDICATOR' in tokens_set:
        candidates.add('TARGET_SPEC_HEMATOLOG')

    # Multi-label: gdy jest specjalista, dopisujemy POZ (pacjent w POZ + skierowanie) = co najmniej 2 etykiety
    if not candidates:
        chosen_labels = {'TARGET_POZ'}
    else:
        chosen_labels = candidates
    label_tokens = " ".join(sorted(chosen_labels))
    seq = " ".join(tokens) + " " + label_tokens
    final_sequences.append(seq)
    for lbl in chosen_labels:
        stats[lbl] = stats.get(lbl, 0) + 1

In [ ]:
import paths_config
paths_config.ensure_dirs()
from datetime import datetime
output_filename = str(paths_config.DATA_PROCESSED / f"synthea_data_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt")
with open(output_filename, "w") as f:
    for s in final_sequences:
        f.write(s + "\n")
        
print(f"✅ Gotowe! Zapisano {len(final_sequences)} sekwencji do {output_filename}.")

In [ ]:
print("--- PRZYKŁADOWE SEKWENCJE ---")
for s in final_sequences[:5]:
    print(s[:100] + "...")

In [ ]:
for line in final_sequences:
    c = line.count("TARGET")
    if c > 1:
        print(line)
        break
